In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
# from BACKTEST.backtest import *
# from MODELS.pipeline import *
from DATA.TOOLS.fetchPlayersStatsV2 import *
from DATA.TOOLS.fetchPlayersStats import *
from DATA.TOOLS.fetchTeamStats import *
from DATA.TOOLS.playerPositions import *

Matplotlib is building the font cache; this may take a moment.


## Fetches Player Gamelogs

In [2]:
nba = FetchPlayersStats()
data = nba.getCompleteStats(
    season='2025-26', 
    season_type='Regular Season', 
    sleep_time=2, 
    max_workers=5,
    batch_limit=50,
    complete_cache_file='../DATA/CSV_FILES/REGULAR_DATA/S26.csv',
    include_playbyplay=False
)
data.tail()

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/DATA/TOOLS/fetchPlayersStats.py:686: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  cache = pd.read_csv(complete_cache_file, dtype={'GAME_ID':str})


Processing 9 games (limited by batch_limit=50)

Processing batch 1/1 (9 games)
Completed batch 1/1

Merging team stats...
Cache updated. Total games now: 222


,Unnamed: 0,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,...,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV_y,OPP_TOV
4912,NaN,Jaden Hardy,1630702,DAL vs. NYK,DAL,1610612742,NYK,1,0022500262,2025-11-19,...,113.0,41.0,93.0,0.441,49.0,27.0,8.0,10.0,NaN,9.0
4913,NaN,Isaac Okoro,1630171,CHI @ POR,CHI,1610612741,POR,0,0022500263,2025-11-19,...,121.0,44.0,107.0,0.411,64.0,27.0,10.0,5.0,NaN,16.0
4914,NaN,Kris Murray,1631200,POR vs. CHI,POR,1610612757,CHI,1,0022500263,2025-11-19,...,122.0,46.0,96.0,0.479,38.0,36.0,10.0,9.0,NaN,16.0
4915,NaN,Josh Giddey,1630581,CHI @ POR,CHI,1610612741,POR,0,0022500263,2025-11-19,...,121.0,44.0,107.0,0.411,64.0,27.0,10.0,5.0,NaN,16.0
4916,NaN,Naji Marshall,1630230,DAL vs. NYK,DAL,1610612742,NYK,1,0022500262,2025-11-19,...,113.0,41.0,93.0,0.441,49.0,27.0,8.0,10.0,NaN,9.0


In [4]:
pd.set_option('display.max_columns', None)

s19_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S19.csv')
s20_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S20.csv')
s21_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S21.csv')
s22_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S22.csv')
s23_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S23.csv')
s24_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S24.csv')
s25_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S25.csv')
s26_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S26.csv')

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_7471/354614631.py:10: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S26.csv')


## Fetching play by play data for model v2

In [ ]:
nba = FetchPlayersStats()
data = nba.getCompleteStats(
    season='2023-24', 
    season_type='Regular Season', 
    sleep_time=2, 
    max_workers=5,
    batch_limit=50,
    complete_cache_file='../DATA/CSV_FILES/REGULAR_DATA/S24v2.csv',
    include_playbyplay=True
)
data.tail()

## Assign features for regular season data

In [6]:
# Add this line after your current imports
from FEATURES.features import teamContext as features_teamContext

def convert_min_to_float(min_str):
    try:
        if isinstance(min_str, str) and ":" in min_str:
            minutes, seconds = map(int, min_str.split(":"))
            total_minutes = minutes + seconds / 60
            return round(total_minutes, 2)
        elif isinstance(min_str, (int, float)):
            return float(min_str)
        else:
            return 0
    except:
        return 0

def process_season_features(season_df, prop_type, year):
    df = season_df.copy()
    df = sort_data_for_features(df)
    df['STARTING'] = df['START_POSITION'].apply(lambda x: 1 if x in ['G','F','C'] else 0)
    df['GAMES_THIS_SEASON'] = df.groupby('PLAYER_ID').cumcount() + 1
    # df['BLOWOUT_RISK'] = (abs(df['spread']) >= 10).astype(int)
    # df['COMPETITIVE_GAME'] = (abs(df['spread']) < 5).astype(int)
    # df['TEAM_IMPLIED_PTS'] = ((df['total'] + df['team_spread']) / 2).round(1)
    
    # Add position data
    cache_file = os.path.join(project_root, 'DATA', 'TOOLS', 'playerInfo.csv')
    df = assign_position_with_cache(df, cache_file=cache_file, max_workers=4, delay_between_requests=1.5)
   
    df = add_rest_day_features(df)
    df['MIN'] = df['MIN'].apply(convert_min_to_float)
    # df, team_mapping = encode_teams_label(df)

    df = rollingAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE', windows=[3,5,10,20,40])
    df = statAgainstTeam(df, player_id_col='PLAYER_ID', opp_col='OPP_ABBREVIATION')
    df = HomeAwayAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE')
    df = getPlayerAvgToDateVectorized(df) 
    df = features_teamContext(df)
    df = assign_opponent_team_stats_dict(df)
    df = assign_team_opp_def_by_position(df, min_minutes=15)
    df = addLagFeatures(df)
    df = add_volatility_features(df, windows=[3, 5, 10, 20,40])
    df = add_recent_form_volatility(df)
    df = process_star_players_data(df, min_minutes=20)
    df = add_performance_without_stars_columns(df, min_games=1)
    df = add_opponent_team_rolling_stats(df, windows=[3,5,10])
    df = add_opponent_team_form_indicators(df, windows=[3,5,10])
    df = expectedPace(df)
    df = add_trend_features(df)
    df = add_player_role_tier_features(df)  
    df = add_interaction_features(df)
    df = add_usual_starters_availability(df)
    df = add_opponent_usual_starters_availability(df)
    df = add_team_rolling_stats(df, windows=[3, 5, 10, 20])
    df = add_opponent_star_status(df)

    # Clean up any unwanted columns
    if 'Unnamed: 0' in df.columns:
        df.drop(columns=['Unnamed: 0'], inplace=True)
    if 'Unnamed: 0.1' in df.columns:
        df.drop(columns=['Unnamed: 0.1'], inplace=True)
    return df

prop_types = ['PTS']
data = [s19_regular,s20_regular, s21_regular,s22_regular,s23_regular,s24_regular, s25_regular, s26_regular]
seasons = [2019,2020,2021,2022,2023,2024, 2025, 2026]
# data = [s26_regular]
# seasons = [2026]

# Pre-define output directory once
output_dir = os.path.join(project_root, 'DATA', 'CSV_FILES', 'TRAIN_DATA')
os.makedirs(output_dir, exist_ok=True)

# Process each season sequentially but with optimized operations
for season_data, year in zip(data, seasons):
    print(f"Processing year {year}...")
    
    # Process features
    processed_data = process_season_features(
        season_data, 
        prop_type='PTS',
        year=year
    )
    
    # Save file
    output_path = os.path.join(output_dir, f'PTS_TRAIN_{str(year)[-2:]}.csv')
    processed_data.to_csv(output_path)
    print(f"Completed {year}")

Processing year 2019...
Loading position cache...
Loaded 1237 players from cache
Found 530 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1237 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!
Completed 2019
Processing year 2020...
Loading position cache...
Loaded 1237 players from cache
Found 529 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1237 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!
Completed 2020
Processing year 2021...
Loading position cache...
Loaded 1237 players from cache
Found 540 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1237 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!
Completed 2021
Processing year 2022...
Loading position cache...
Loaded 1237 players from cache
Found 605 unique players, 0 need to be fetched
Saving u